<a href="https://colab.research.google.com/github/areslou/cpecog2-coral-bleaching/blob/main/notebooks/Group2_Coral_Run10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 2 CPECOG2: Coral bleaching, Run 10 (knowledge distillation from DINOv2 into YOLOv8s-cls)

Runtime > Change runtime type > **T4 GPU**, then Runtime > Run all (about 50 to 60 min).

**The deployed model is still only YOLOv8s-cls** (ImageNet weights, 2-class output, 10.3 MB, Grad-CAM heatmap in the demo). What changes is how it is trained. Run 9 showed that a fine-tuned DINOv2 ViT-S/14 ranks photos better in cross-validation (macro F1 0.853 vs. 0.801) and on the external BU + BHD set, but it is 4 times larger, slower, and has no heatmap in the demo. Knowledge distillation (Hinton, Vinyals and Dean, 2015) trains the small model to match both the folder labels and the larger model's softened predictions:

loss = α · cross-entropy(student, label) + (1 − α) · T² · KL(teacher softened by T ‖ student softened by T)

- **Teacher:** DINOv2 ViT-S/14, fine-tuned as in Run 9. In each fold it is trained **only on that fold's training photos**; the final teacher is trained only on the 761 training + validation photos. It never sees the held-out fold or the test set. It is discarded after training.
- **Student:** YOLOv8s-cls with the Run 7 settings (dropout 0.3, crops of 75 to 100% of the area, flips, small brightness, contrast and saturation shifts, AdamW lr0 0.00167 with a warm-up and linear decay, EMA weights, 20 epochs). Distillation needs a custom PyTorch training loop around the Ultralytics model, so **S0 trains the student in the same loop without a teacher** as the control: any gain over S0 is the teacher's.
- **Tuned distillation settings (added to the hyperparameter register):** α (weight on the labels) and T (temperature).

| Config | α | T |
| :-- | :-- | :-- |
| S0 (control, no teacher) | 1.0 | n/a |
| K1 | 0.5 | 2 |
| K2 | 0.5 | 4 |
| K3 | 0.2 | 4 |

All configs use the **same 5 folds** as Runs 6 to 9 and seed 42. The best one is then trained with seeds 43 and 44 so the 3-seed average can be compared with the Run 7 final model (3-seed YOLOv8s-cls average, out-of-fold macro F1 **0.801** at its recall-floor cutoff). **The test set is used once, and only if it beats Run 7.** The saved students are standard Ultralytics checkpoints, so `app.py` loads them unchanged.

In [ ]:
!pip -q install ultralytics==8.4.211 imagehash kagglehub huggingface_hub

In [ ]:
import os, glob, shutil, time, json, random, math, re, zipfile, copy, warnings
from datetime import datetime
import numpy as np, pandas as pd
from PIL import Image
import imagehash, torch
import torch.nn as nn, torch.nn.functional as F
import torchvision.transforms.v2 as T2
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, roc_auc_score, roc_curve, confusion_matrix
warnings.filterwarnings("ignore", category=DeprecationWarning)

SEED = 42
FOLDS = 5
EPOCHS = 20                  # student, as in Run 7
DINO_EPOCHS = 15             # teacher, as in Run 9
BATCH = 32
IMGSZ = 224
SEEDS = [42, 43, 44]
LR0 = 0.002 * 5 / (4 + 2)    # what Ultralytics' optimizer="auto" picks for 2 classes (AdamW, about 0.00167)
GRID = {"S0_no_teacher": (1.0, None), "K1_a0.5_T2": (0.5, 2.0), "K2_a0.5_T4": (0.5, 4.0), "K3_a0.2_T4": (0.2, 4.0)}
RECALL_MIN = 0.90
MAX_MB, MAX_MS = 50, 200
NEAR_DUP_T = 5
RUN7_OOF_MACRO_F1, RUN7_OOF_AUC = 0.801, 0.912     # Run 7 final (3-seed YOLOv8s-cls average), same folds
RUN7_SINGLE_OOF_MACRO_F1, RUN7_SINGLE_OOF_AUC = 0.788, 0.894
EVALUATE_TEST = True         # the test set is still used only if the distilled model beats Run 7 in cross-validation
DATA_SRC = None
EXT_SRC = None
SPLITS_URL = "https://raw.githubusercontent.com/areslou/cpecog2-coral-bleaching/main/data/splits.csv"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
TDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
print("GPU:", GPU_NAME if torch.cuda.is_available() else "none. Runtime > Change runtime type > T4 GPU, or this will take hours.")
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
OUT = os.path.join(BASE, "results", "run10"); os.makedirs(OUT, exist_ok=True)
CKPT = os.path.join(BASE, "runs10"); os.makedirs(CKPT, exist_ok=True)
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
T_START = time.time()

## 1. Data: the cleaned split and the Run 6 folds

In [ ]:
if DATA_SRC is None:
    import kagglehub
    DATA_SRC = kagglehub.dataset_download("vencerlanz09/healthy-and-bleached-corals-image-classification")
exts = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def label_of(p):
    parent = os.path.basename(os.path.dirname(p)).lower()
    if "bleach" in parent: return "bleached"
    if "health" in parent: return "healthy"
    return None

def phash_of(p):
    try:
        with Image.open(p) as im:
            return str(imagehash.phash(im.convert("RGB")))
    except Exception:
        return None

POP = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def hamming(a, b):
    x = np.array([int(h, 16) for h in a], dtype=np.uint64)[:, None] ^ np.array([int(h, 16) for h in b], dtype=np.uint64)[None, :]
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x).astype(np.int16)
    return POP[np.ascontiguousarray(x).view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1).astype(np.int16)

paths = sorted(p for p in glob.glob(os.path.join(DATA_SRC, "**", "*"), recursive=True) if p.lower().endswith(exts))
df = pd.DataFrame({"path": paths})
df["label"] = df["path"].map(label_of)
df["file"] = df["path"].map(os.path.basename)
df["phash"] = df["path"].map(phash_of)
df = df.dropna(subset=["label", "phash"]).reset_index(drop=True)

splits1 = pd.read_csv(SPLITS_URL).rename(columns={"path": "file"})
df = df.merge(splits1[["file", "label", "split"]], on=["file", "label"], how="left")
assert int(df["split"].notna().sum()) == len(splits1), "splits.csv did not match the download"

n_lab = df.groupby("phash")["label"].nunique()
conflict_hashes = set(n_lab[n_lab > 1].index)
clean = df[df["split"].notna() & ~df["phash"].isin(conflict_hashes)].reset_index(drop=True)
D = hamming(clean["phash"], clean["phash"])
sp, lb = clean["split"].to_numpy(), clean["label"].to_numpy()
ii, jj = np.where(np.triu(D <= NEAR_DUP_T, k=1))
rank = {"train": 0, "val": 1, "test": 2}
drop = {max((i, j), key=lambda k: rank[sp[k]]) for i, j in zip(ii, jj) if sp[i] != sp[j] and lb[i] == lb[j]}
drop |= {k for i, j in zip(ii, jj) if lb[i] != lb[j] for k in (i, j)}
clean = clean.drop(index=sorted(drop)).reset_index(drop=True)
print(f"Removed {len(conflict_hashes)} conflicting-label groups and {len(drop)} near-duplicates")
print(pd.crosstab(clean["split"], clean["label"], margins=True))

In [ ]:
def pad_square(p):
    im = Image.open(p).convert("RGB"); s = max(im.size)
    sq = Image.new("RGB", (s, s), (0, 0, 0)); sq.paste(im, ((s - im.size[0]) // 2, (s - im.size[1]) // 2))
    return sq

POOL = os.path.join(BASE, "coral_pool"); shutil.rmtree(POOL, ignore_errors=True)
imgs = []
for _, r in clean.iterrows():
    d = os.path.join(POOL, r["label"]); os.makedirs(d, exist_ok=True)
    dst = os.path.join(d, os.path.splitext(r["file"])[0] + ".jpg")
    pad_square(r["path"]).save(dst, quality=95); imgs.append(dst)
clean["img"] = imgs
assert clean["img"].nunique() == len(clean), "file name clash in the image pool"

trval = clean[clean["split"] != "test"].reset_index(drop=True)
test = clean[clean["split"] == "test"].reset_index(drop=True)
y_trval = (trval["label"] == "bleached").astype(int).to_numpy()
y_test = (test["label"] == "bleached").astype(int).to_numpy()
test_files = test["img"].tolist()
FOLD_IDX = list(StratifiedKFold(FOLDS, shuffle=True, random_state=SEED).split(np.zeros(len(trval)), y_trval))
fold_of = np.zeros(len(trval), dtype=int)
for k, (_, va) in enumerate(FOLD_IDX): fold_of[va] = k
trval.assign(fold=fold_of)[["file", "label", "split", "fold"]].to_csv(os.path.join(OUT, "folds.csv"), index=False)
test[["file", "label"]].to_csv(os.path.join(OUT, "test_images.csv"), index=False)
print(f"Train + validation: {len(trval)} ({y_trval.sum()} bleached) in {FOLDS} folds | test: {len(test)} ({y_test.sum()} bleached)")

## 2. Helpers: metrics, the teacher (Run 9 code), and the student training loop

The student is the Ultralytics YOLOv8s-cls network itself (`YOLO("yolov8s-cls.pt").model`) with its output layer reshaped to 2 classes. Training uses mixed precision, AdamW with weight decay 0.0005 on weights only, a warm-up over the first max(3 epochs, 100 steps), a linear decay to 1%, and an exponential moving average of the weights (all as in Ultralytics). Each trained student is saved as a standard Ultralytics checkpoint and scored with `YOLO(...).predict`, the same path the demo uses.

In [ ]:
def metrics_at(y, p, thr):
    yp = (np.asarray(p) >= thr).astype(int)
    return dict(macro_f1=f1_score(y, yp, average="macro"), bleached_recall=recall_score(y, yp, zero_division=0),
                bleached_precision=precision_score(y, yp, zero_division=0), healthy_recall=recall_score(y, yp, pos_label=0, zero_division=0),
                accuracy=accuracy_score(y, yp), roc_auc=roc_auc_score(y, p), cm=confusion_matrix(y, yp, labels=[0, 1]).tolist())

def pick_threshold(y, p):
    rows = [(t, f1_score(y, p >= t, average="macro"), recall_score(y, p >= t, zero_division=0)) for t in np.round(np.linspace(0.01, 0.99, 981), 3)]
    ok = [r for r in rows if r[2] >= RECALL_MIN]
    return max(ok or rows, key=lambda r: (r[1], r[0]))[0], bool(ok)

def make_folder(name, train_idx, val_idx):
    root = os.path.join(BASE, "cv", name); shutil.rmtree(root, ignore_errors=True)
    for part, idx in (("train", train_idx), ("val", val_idx)):
        for i in idx:
            r = trval.iloc[i]; d = os.path.join(root, part, r["label"]); os.makedirs(d, exist_ok=True)
            os.symlink(r["img"], os.path.join(d, os.path.basename(r["img"])))
    return root

from ultralytics import YOLO
def yolo_fit(data_root, name, seed):
    m = YOLO("yolov8s-cls.pt")
    m.train(data=data_root, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=seed, deterministic=True, patience=EPOCHS + 1,
            val=False, plots=False, project=os.path.join(BASE, "runs7"), name=name, exist_ok=True, device=DEVICE, verbose=False, **YOLO_AUG)
    return str(m.trainer.last)

def yolo_predict(weights, files, bs=64):
    m = YOLO(weights); b = next(k for k, n in m.names.items() if n == "bleached"); p = []
    for k in range(0, len(files), bs):
        p += [float(r.probs.data[b]) for r in m.predict(files[k:k + bs], imgsz=IMGSZ, device=DEVICE, verbose=False)]
    return np.array(p)

IMNET_MEAN, IMNET_STD = torch.tensor([0.485, 0.456, 0.406]), torch.tensor([0.229, 0.224, 0.225])

def load_dino_backbone():
    return torch.hub.load("facebookresearch/dinov2", "dinov2_vits14", verbose=False)

class DinoClassifier(nn.Module):
    def __init__(self, backbone, n_classes=2, p=0.3):
        super().__init__()
        self.backbone, self.drop = backbone, nn.Dropout(p)
        self.head = nn.Linear(2 * backbone.embed_dim, n_classes)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        return self.head(self.drop(torch.cat([f["x_norm_clstoken"], f["x_norm_patchtokens"].mean(1)], 1)))

_tensors = {}
def load_tensor(files, size):
    return torch.stack([torch.from_numpy(np.array(Image.open(p).convert("RGB").resize((size, size), Image.BILINEAR))).permute(2, 0, 1)
                        for p in files])

def trval_tensor(size):
    if size not in _tensors: _tensors[size] = load_tensor(trval["img"].tolist(), size)
    return _tensors[size]

def norm(xb):
    return (xb.float() / 255 - IMNET_MEAN.to(xb.device)[:, None, None]) / IMNET_STD.to(xb.device)[:, None, None]

def augment(xb):
    if random.random() < 0.5: xb = xb.flip(3)
    if random.random() < 0.5: xb = xb.flip(2)
    n = xb.shape[-1]; c = int(n * random.uniform(0.87, 1.0)); i, j = random.randint(0, n - c), random.randint(0, n - c)
    return F.interpolate(xb[..., i:i + c, j:j + c], size=(n, n), mode="bilinear", align_corners=False)

def dino_fit(train_idx, seed, size):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    X, y = trval_tensor(size)[train_idx], torch.as_tensor(y_trval[train_idx])
    net = DinoClassifier(load_dino_backbone()).to(TDEV)
    opt = torch.optim.AdamW([{"params": net.backbone.parameters(), "lr": 1e-5}, {"params": net.head.parameters(), "lr": 1e-3}], weight_decay=0.05)
    steps_per = math.ceil(len(X) / BATCH); total = DINO_EPOCHS * steps_per
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / steps_per) * 0.5 * (1 + math.cos(math.pi * s / total)))
    use_amp = TDEV.type == "cuda"
    scaler = torch.amp.GradScaler(enabled=use_amp)
    for ep in range(DINO_EPOCHS):
        net.train(); perm = torch.randperm(len(X))
        for k in range(0, len(perm), BATCH):
            idx = perm[k:k + BATCH]
            if len(idx) < 2: continue
            xb = augment(norm(X[idx].to(TDEV)))
            with torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=use_amp):
                loss = F.cross_entropy(net(xb), y[idx].to(TDEV))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    return net.eval()

def dino_predict(net, X, bs=64):
    out = []
    with torch.no_grad(), torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=TDEV.type == "cuda"):
        for k in range(0, len(X), bs):
            out.append(net(norm(X[k:k + bs].to(TDEV))).float().softmax(1)[:, 1].cpu())
    return torch.cat(out).numpy()

from ultralytics.nn.tasks import ClassificationModel
import ultralytics

AUG = T2.Compose([T2.RandomResizedCrop(IMGSZ, scale=(0.75, 1.0), antialias=True), T2.RandomHorizontalFlip(), T2.RandomVerticalFlip(),
                  T2.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.0)])
X224 = trval_tensor(IMGSZ)

def new_student():
    net = YOLO("yolov8s-cls.pt").model
    ClassificationModel.reshape_outputs(net, 2)
    net.names = {0: "healthy", 1: "bleached"}
    for m in net.modules():
        if isinstance(m, nn.Dropout): m.p = 0.3
    for p in net.parameters(): p.requires_grad_(True)
    return net.float()

class EMA:
    """Exponential moving average of the weights, as in Ultralytics (decay 0.9999, tau 2000)."""
    def __init__(self, model):
        self.ema, self.updates = copy.deepcopy(model).eval(), 0
        for p in self.ema.parameters(): p.requires_grad_(False)
    def update(self, model):
        self.updates += 1; d = 0.9999 * (1 - math.exp(-self.updates / 2000))
        msd = model.state_dict()
        with torch.no_grad():
            for k, v in self.ema.state_dict().items():
                if v.dtype.is_floating_point: v.mul_(d).add_(msd[k].detach(), alpha=1 - d)

def student_fit(idx, seed, alpha, temp, teacher=None):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    X, y = X224[idx], torch.as_tensor(y_trval[idx])
    net = new_student().to(TDEV); ema = EMA(net)
    decay = [p for p in net.parameters() if p.ndim > 1]; no_decay = [p for p in net.parameters() if p.ndim <= 1]
    opt = torch.optim.AdamW([{"params": decay, "weight_decay": 5e-4}, {"params": no_decay, "weight_decay": 0.0}], lr=LR0, betas=(0.9, 0.999))
    nb = math.ceil(len(X) / BATCH); warm = max(3 * nb, 100); it = 0
    use_amp = TDEV.type == "cuda"; scaler = torch.amp.GradScaler(enabled=use_amp)
    for ep in range(EPOCHS):
        net.train(); perm = torch.randperm(len(X))
        for k in range(0, len(perm), BATCH):
            b = perm[k:k + BATCH]
            if len(b) < 2: continue
            xb = torch.stack([AUG(img) for img in X[b].to(TDEV).float() / 255])
            lr = LR0 * ((1 - ep / EPOCHS) * 0.99 + 0.01) * min(1.0, (it + 1) / warm)
            for g in opt.param_groups: g["lr"] = lr
            with torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=use_amp):
                s = net(xb)
            s = s.float(); loss = alpha * F.cross_entropy(s, y[b].to(TDEV))
            if alpha < 1:
                with torch.no_grad(), torch.autocast(device_type=TDEV.type, dtype=torch.float16, enabled=use_amp):
                    t = teacher((xb - IMNET_MEAN.to(TDEV)[:, None, None]) / IMNET_STD.to(TDEV)[:, None, None])
                loss = loss + (1 - alpha) * temp ** 2 * F.kl_div(F.log_softmax(s / temp, 1), F.softmax(t.float() / temp, 1), reduction="batchmean")
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); ema.update(net); it += 1
    return ema.ema

def save_student(net, path):
    m = copy.deepcopy(net).half().eval()
    for p in m.parameters(): p.requires_grad_(False)
    torch.save({"model": m, "train_args": {"task": "classify", "imgsz": IMGSZ, "model": "yolov8s-cls.pt"},
                "date": datetime.now().isoformat(), "version": ultralytics.__version__, "epoch": -1}, path)
    return path

## 3. Cross-validation: teacher per fold, then S0 and K1 to K3 (seed 42)

Progress prints after each fold.

In [ ]:
FOLD_ROOTS = None
oof = {n: np.full(len(trval), np.nan) for n in GRID}
teacher_auc, TEACHERS, t0 = [], {}, time.time()
for k, (tr, va) in enumerate(FOLD_IDX):
    teacher = dino_fit(tr, SEED, IMGSZ)
    teacher_auc.append(roc_auc_score(y_trval[va], dino_predict(teacher, X224[va])))
    TEACHERS[k] = {n: v.detach().half().cpu() for n, v in teacher.state_dict().items()}
    for name, (alpha, temp) in GRID.items():
        stu = student_fit(tr, SEED, alpha, temp, teacher if alpha < 1 else None)
        oof[name][va] = yolo_predict(save_student(stu, os.path.join(CKPT, f"{name}_f{k}_s{SEED}.pt")), trval["img"].iloc[va].tolist())
        del stu
    del teacher; torch.cuda.empty_cache()
    print(f">>> fold {k + 1}/{FOLDS}: teacher ROC AUC {teacher_auc[-1]:.3f}; " + ", ".join(f"{n} {roc_auc_score(y_trval[va], p[va]):.3f}" for n, p in oof.items())
          + f" ({(time.time() - t0) / 60:.1f} min)")

def load_teacher(k):
    t = DinoClassifier(load_dino_backbone()); t.load_state_dict({n: v.float() for n, v in TEACHERS[k].items()}); return t.to(TDEV).eval()

## 4. Results, then the best config with 3 seeds

In [ ]:
def summarize(name, p):
    thr, reached = pick_threshold(y_trval, p); mt = metrics_at(y_trval, p, thr)
    return dict(config=name, oof_roc_auc=roc_auc_score(y_trval, p), macro_f1_at_0_5=f1_score(y_trval, p >= 0.5, average="macro"), cutoff=thr,
                recall_floor_reached=reached, macro_f1_at_cutoff=mt["macro_f1"], recall_at_cutoff=mt["bleached_recall"], healthy_recall_at_cutoff=mt["healthy_recall"])
grid = pd.DataFrame([summarize(n, p) for n, p in oof.items()]).sort_values(["recall_floor_reached", "macro_f1_at_cutoff", "oof_roc_auc"], ascending=False)
print(grid.round(3).to_string(index=False))
print(f"Teacher (DINOv2) out-of-fold ROC AUC by fold: {np.round(teacher_auc, 3).tolist()}")
WIN = grid.iloc[0]["config"]; alpha, temp = GRID[WIN]
print(f"\nBest config: {WIN}. Training seeds {SEEDS[1:]} on every fold for the 3-seed average...")

t0 = time.time(); seed_oof = {SEED: oof[WIN]}
for s in SEEDS[1:]:
    p = np.full(len(trval), np.nan)
    for k, (tr, va) in enumerate(FOLD_IDX):
        teacher = load_teacher(k) if alpha < 1 else None
        stu = student_fit(tr, s, alpha, temp, teacher)
        p[va] = yolo_predict(save_student(stu, os.path.join(CKPT, f"{WIN}_f{k}_s{s}.pt")), trval["img"].iloc[va].tolist())
        del stu, teacher; torch.cuda.empty_cache()
        print(f">>> {WIN} seed {s}, fold {k + 1}/{FOLDS} ({(time.time() - t0) / 60:.1f} min)")
    seed_oof[s] = p
ENS = f"{WIN}_3seed_average"; oof[ENS] = np.mean([seed_oof[s] for s in SEEDS], axis=0)

cv = pd.DataFrame([summarize(n, p) for n, p in oof.items()])
cv = pd.concat([cv, pd.DataFrame([dict(config="Run 7 YOLOv8s-cls single (Ultralytics train)", oof_roc_auc=RUN7_SINGLE_OOF_AUC, macro_f1_at_cutoff=RUN7_SINGLE_OOF_MACRO_F1),
                                  dict(config="Run 7 final, 3-seed average", oof_roc_auc=RUN7_OOF_AUC, macro_f1_at_cutoff=RUN7_OOF_MACRO_F1)])], ignore_index=True)
cv.to_csv(os.path.join(OUT, "cv_results.csv"), index=False)
pd.DataFrame({"file": trval["file"], "true_label": y_trval, **oof, **{f"{WIN}_s{s}": v for s, v in seed_oof.items()}}).to_csv(os.path.join(OUT, "oof_predictions.csv"), index=False)
print(cv.round(3).to_string(index=False))

c = cv.set_index("config")
USE_ENS = c.loc[ENS, "macro_f1_at_cutoff"] > c.loc[WIN, "macro_f1_at_cutoff"]
FINAL_NAME = ENS if USE_ENS else WIN
THR, THR_SINGLE = float(c.loc[FINAL_NAME, "cutoff"]), float(c.loc[WIN, "cutoff"])
BEATS = bool(c.loc[FINAL_NAME, "macro_f1_at_cutoff"] > RUN7_OOF_MACRO_F1)
CHANGE = BEATS and EVALUATE_TEST
print(f"\nBest Run 10 model: {FINAL_NAME} (out-of-fold macro F1 {c.loc[FINAL_NAME, 'macro_f1_at_cutoff']:.3f} vs. Run 7 {RUN7_OOF_MACRO_F1:.3f}); "
      f"beats Run 7: {BEATS}. " + ("Continuing to the final models and the test set." if CHANGE else "The test set is NOT used; Run 7 stays the final model."))

fig, ax = plt.subplots(figsize=(5.4, 4.6)); markers = ["o", "s", "^", "D", "v", "P"]
for i, (n, p) in enumerate(oof.items()):
    fpr, tpr, _ = roc_curve(y_trval, p)
    ax.plot(fpr, tpr, color=COLORS[i % 6], lw=2, marker=markers[i % 6], markevery=0.1, ms=5, label=f"{n} (AUC {roc_auc_score(y_trval, p):.3f})")
ax.plot([0, 1], [0, 1], color="#888888", lw=1, ls="--"); ax.axhline(RECALL_MIN, color="#888888", lw=1, ls=":")
ax.set_xlabel("False positive rate (healthy flagged as bleached)"); ax.set_ylabel("Bleached recall")
ax.set_title(f"Run 10 out-of-fold ROC, {FOLDS}-fold CV (n={len(trval)})", fontsize=10)
ax.grid(alpha=0.25); ax.spines[["top", "right"]].set_visible(False); ax.legend(fontsize=7, frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "cv_roc.png"), dpi=150); plt.show()

## 5. Final models (only if Run 10 beats Run 7): teacher and 3 students on all 761 images

In [ ]:
WEIGHTS = {}
if CHANGE:
    teacher = dino_fit(np.arange(len(trval)), SEED, IMGSZ) if alpha < 1 else None
    for s in SEEDS:
        t0 = time.time()
        WEIGHTS[s] = save_student(student_fit(np.arange(len(trval)), s, alpha, temp, teacher), os.path.join(OUT, f"final_model_s{s}.pt"))
        print(f">>> final student seed {s}: {(time.time() - t0) / 60:.1f} min")
    del teacher; torch.cuda.empty_cache()

## 6. Held-out test set and external BU + BHD set (only if Run 10 beats Run 7)

Scored through `YOLO(...).predict`, exactly as the demo loads the model. Each seed is scored at the single-model cutoff, the 3-seed average at its own cutoff, and everything also at 0.5.

In [ ]:
EXT = None
if CHANGE:
    p_seed = {s: yolo_predict(WEIGHTS[s], test_files) for s in SEEDS}; p_ens = np.mean([p_seed[s] for s in SEEDS], axis=0)
    rows = []
    for s in SEEDS:
        rows += [dict(model=f"seed {s}", at="cutoff", threshold=THR_SINGLE, **metrics_at(y_test, p_seed[s], THR_SINGLE)),
                 dict(model=f"seed {s}", at="0.5", threshold=0.5, **metrics_at(y_test, p_seed[s], 0.5))]
    rows += [dict(model="3-seed average", at="cutoff", threshold=THR, **metrics_at(y_test, p_ens, THR)),
             dict(model="3-seed average", at="0.5", threshold=0.5, **metrics_at(y_test, p_ens, 0.5))]
    test_res = pd.DataFrame(rows); test_res.to_csv(os.path.join(OUT, "test_results.csv"), index=False)
    pd.DataFrame(dict(file=test["file"], true_label=y_test, **{f"p_seed{s}": p_seed[s] for s in SEEDS}, p_average=p_ens)).to_csv(os.path.join(OUT, "test_predictions.csv"), index=False)
    print(test_res.drop(columns=["cm"]).round(3).to_string(index=False))
    ms = lambda f, col: (f[col].mean(), f[col].std(ddof=1))
    seeds_cut = test_res[test_res["model"].str.startswith("seed") & (test_res["at"] == "cutoff")]
    seeds_05 = test_res[test_res["model"].str.startswith("seed") & (test_res["at"] == "0.5")]
    ENS_CUT = test_res[(test_res["model"] == "3-seed average") & (test_res["at"] == "cutoff")].iloc[0]
    ENS_05 = test_res[(test_res["model"] == "3-seed average") & (test_res["at"] == "0.5")].iloc[0]

In [ ]:
if CHANGE:
    try:
        if EXT_SRC is None:
            from huggingface_hub import hf_hub_download
            EXT_SRC = hf_hub_download("esahit/coral-health-classification", "total_dataset.zip", repo_type="dataset")
        raw = os.path.join(BASE, "ext_raw"); shutil.rmtree(raw, ignore_errors=True)
        with zipfile.ZipFile(EXT_SRC) as z: z.extractall(raw)
        def parse(name):
            s = name.lower()
            lab = "bleached" if "unhealthy" in s else "healthy" if "healthy" in s else "dead" if "dead" in s else None
            src = "BHD" if ("hbd" in s or "bhd" in s) else "BU" if re.search(r"(?<![a-z])bu(?![a-z])", s) else "EILAT" if "eilat" in s else None
            return lab, src
        files = sorted(p for p in glob.glob(os.path.join(raw, "**", "*"), recursive=True) if p.lower().endswith(exts + (".tif", ".tiff")))
        ext = pd.DataFrame([(p, *parse(os.path.basename(p))) for p in files], columns=["path", "label", "source"])
        print(pd.crosstab(ext["source"].fillna("unknown"), ext["label"].fillna("unknown")))
        ext = ext[ext["label"].isin(["healthy", "bleached"]) & ext["source"].isin(["BU", "BHD"])].reset_index(drop=True)
        ext["phash"] = ext["path"].map(phash_of); ext = ext.dropna(subset=["phash"])
        n_raw = len(ext); ext = ext.drop_duplicates("phash").reset_index(drop=True); n_within = n_raw - len(ext)
        overlap = hamming(ext["phash"], df["phash"]).min(1) <= NEAR_DUP_T
        ext = ext[~overlap].reset_index(drop=True)
        sq = os.path.join(BASE, "ext_sq"); shutil.rmtree(sq, ignore_errors=True); os.makedirs(sq)
        ext["img"] = [os.path.join(sq, f"{r.source}_{i:05d}.jpg") for i, r in enumerate(ext.itertuples())]
        for r in ext.itertuples(): pad_square(r.path).save(r.img, quality=95)
        y_ext = (ext["label"] == "bleached").astype(int).to_numpy(); src = ext["source"].to_numpy()
        pe = {s: yolo_predict(WEIGHTS[s], ext["img"].tolist()) for s in SEEDS}; pe_ens = np.mean([pe[s] for s in SEEDS], axis=0)
        rows = []
        for name, mask in (("BU + BHD", np.ones(len(y_ext), bool)), ("BU", src == "BU"), ("BHD", src == "BHD")):
            for s in SEEDS:
                rows.append(dict(set=name, n=int(mask.sum()), model=f"seed {s}", threshold=THR_SINGLE, **metrics_at(y_ext[mask], pe[s][mask], THR_SINGLE)))
            rows.append(dict(set=name, n=int(mask.sum()), model="3-seed average", threshold=THR, **metrics_at(y_ext[mask], pe_ens[mask], THR)))
            rows.append(dict(set=name, n=int(mask.sum()), model="3-seed average @0.5", threshold=0.5, **metrics_at(y_ext[mask], pe_ens[mask], 0.5)))
        EXT = pd.DataFrame(rows); EXT.to_csv(os.path.join(OUT, "external_results.csv"), index=False)
        EXT_INFO = dict(n=len(y_ext), bleached=int(y_ext.sum()), healthy=int((y_ext == 0).sum()), removed_within=n_within, removed_overlap=int(overlap.sum()),
                        by_source=ext["source"].value_counts().to_dict())
        print(EXT_INFO); print(EXT.drop(columns=["cm"]).round(3).to_string(index=False))
    except Exception as e:
        print("External test skipped:", repr(e))

## 7. Size and CPU latency

Mean over 100 test images, batch 1, after 10 warm-ups, image loading included, on Colab's CPU (repeat on a team laptop for the final report).

In [ ]:
if CHANGE:
    cpu_models = {s: YOLO(WEIGHTS[s]) for s in SEEDS}
    LAT, key = {}, ("3-seed average" if USE_ENS else "single")
    for label, seeds in (("single", [SEEDS[0]]), ("3-seed average", SEEDS)):
        one = lambda p: [cpu_models[s].predict(p, imgsz=IMGSZ, device="cpu", verbose=False) for s in seeds]
        for p in test_files[:10]: one(p)
        t0 = time.perf_counter()
        for p in test_files[:100]: one(p)
        LAT[label] = (time.perf_counter() - t0) / len(test_files[:100]) * 1000
    SIZE = {"single": os.path.getsize(WEIGHTS[SEEDS[0]]) / 1e6, "3-seed average": sum(os.path.getsize(WEIGHTS[s]) for s in SEEDS) / 1e6}
    card = dict(run=10, final=FINAL_NAME, architecture="YOLOv8s-cls (Ultralytics), 2-class output, dropout 0.3",
                training=f"knowledge distillation from DINOv2 ViT-S/14 (alpha={alpha}, T={temp})" if alpha < 1 else "custom loop, no teacher",
                models=[os.path.basename(WEIGHTS[s]) for s in (SEEDS if USE_ENS else [SEEDS[0]])], combine="mean of P(bleached)" if USE_ENS else "single model",
                threshold=THR if USE_ENS else THR_SINGLE, threshold_source=f"{FOLDS}-fold out-of-fold predictions (recall >= {RECALL_MIN})",
                classes={"0": "healthy", "1": "bleached"}, imgsz=IMGSZ, size_mb={k: round(v, 1) for k, v in SIZE.items()}, cpu_latency_ms={k: round(v) for k, v in LAT.items()})
    json.dump(card, open(os.path.join(OUT, "final_model.json"), "w"), indent=2)
    print("Size MB:", {k: round(v, 1) for k, v in SIZE.items()}, "| CPU latency ms:", {k: round(v) for k, v in LAT.items()})

## 8. Summary for the progress log

In [ ]:
f3 = lambda m: f"{m[0]:.3f} ± {m[1]:.3f}"
ok = lambda b: "met" if b else "NOT met"
L = [f"RUN 10 SUMMARY ({time.strftime('%b. %d, %Y')})",
     f"Distillation from DINOv2 ViT-S/14 into YOLOv8s-cls; teacher trained per fold on training photos only; {FOLDS}-fold CV on train + validation (n={len(trval)}).",
     f"Teacher out-of-fold ROC AUC by fold: {np.round(teacher_auc, 3).tolist()}.",
     "Cross-validation (out-of-fold):"]
for r in cv.itertuples():
    L.append(f"  {r.config}: ROC AUC {r.oof_roc_auc:.3f}" + (f"; macro F1 @0.5 {r.macro_f1_at_0_5:.3f}; cutoff {r.cutoff:.3f} -> macro F1 {r.macro_f1_at_cutoff:.3f}, "
             f"bleached recall {r.recall_at_cutoff:.3f}, healthy recall {r.healthy_recall_at_cutoff:.3f}" if not pd.isna(r.cutoff) else f"; macro F1 at its cutoff {r.macro_f1_at_cutoff:.3f}"))
L.append(f"Best config: {WIN} (alpha={alpha}, T={temp}); final form: {FINAL_NAME}; beats Run 7 in cross-validation: {BEATS}; test set evaluated: {CHANGE}.")
if CHANGE:
    L += [f"Held-out test (n={len(test)}, evaluated once):",
          f"  Single students, seeds {SEEDS}, at cutoff {THR_SINGLE:.3f}: macro F1 {f3(ms(seeds_cut, 'macro_f1'))}, bleached recall {f3(ms(seeds_cut, 'bleached_recall'))}, "
          f"healthy recall {f3(ms(seeds_cut, 'healthy_recall'))}, ROC AUC {f3(ms(seeds_cut, 'roc_auc'))}; at 0.5 macro F1 {f3(ms(seeds_05, 'macro_f1'))}, recall {f3(ms(seeds_05, 'bleached_recall'))}.",
          f"  3-seed average at cutoff {THR:.3f}: macro F1 {ENS_CUT.macro_f1:.3f}, bleached recall {ENS_CUT.bleached_recall:.3f}, healthy recall {ENS_CUT.healthy_recall:.3f}, "
          f"ROC AUC {ENS_CUT.roc_auc:.3f}, confusion matrix {ENS_CUT.cm}; at 0.5 macro F1 {ENS_05.macro_f1:.3f}, recall {ENS_05.bleached_recall:.3f}."]
    if EXT is not None:
        for setname in ("BU + BHD", "BU", "BHD"):
            g = EXT[EXT["set"] == setname]; sd = g[g["model"].str.startswith("seed")]; e = g[g["model"] == "3-seed average"].iloc[0]
            L.append(f"  External {setname} (n={int(g['n'].iloc[0])}): single students at cutoff macro F1 {f3(ms(sd, 'macro_f1'))}, recall {f3(ms(sd, 'bleached_recall'))}; "
                     f"3-seed average at cutoff macro F1 {e.macro_f1:.3f}, recall {e.bleached_recall:.3f}.")
    mf1 = ENS_CUT.macro_f1 if USE_ENS else ms(seeds_cut, "macro_f1")[0]; mrec = ENS_CUT.bleached_recall if USE_ENS else ms(seeds_cut, "bleached_recall")[0]
    L += [f"Final model ({key}): {SIZE[key]:.1f} MB, {LAT[key]:.0f} ms/image (Colab CPU); single student {SIZE['single']:.1f} MB, {LAT['single']:.0f} ms.",
          f"Targets on test (final model at its cutoff): macro F1 >= 0.90 {ok(mf1 >= 0.90)} ({mf1:.3f}); bleached recall >= 0.90 {ok(mrec >= 0.90)} ({mrec:.3f}); "
          f"size < 50 MB {ok(SIZE[key] < MAX_MB)}; latency < 200 ms {ok(LAT[key] < MAX_MS)}."]
else:
    L.append("Test set not used; Run 7 (YOLOv8s-cls, 3-seed average) stays the final model.")
L.append(f"Environment: GPU {GPU_NAME}, ultralytics {ultralytics.__version__}, torch {torch.__version__}. Total time {(time.time() - T_START) / 60:.1f} min.")
summary = "\n".join(L); print(summary); open(os.path.join(OUT, "run10_summary.txt"), "w").write(summary)
ZIP = shutil.make_archive(os.path.join(BASE, "run10_results"), "zip", OUT); print("\nSaved", ZIP)
try:
    from google.colab import files
    files.download(ZIP)
except Exception:
    pass